In [1]:
from ETL.ingestion import data_ingestion_hdma
import os
import sqlite3
from pathlib import Path
import pandas as pd

temp_num_cols = [
    'activity_year', 'action_taken', 'preapproval', 'loan_purpose', 'loan_amount', 'loan_to_value_ratio',
    'loan_term', 'income', 'debt_to_income_ratio'
    ]
temp_text_cols = [
    'derived_loan_product_type', 'applicant_credit_score_type', 'co-applicant_credit_score_type', 'denial_reason-1'
    ]

DATA = data_ingestion_hdma.initialize_data_path()

print(DATA)

C:\Users\Saul\Desktop\loan_def_pred\src\training_data


In [2]:
hdma_accepted_parquet = DATA / 'hdma_accepted_raw.parquet.gzip'
hdma_rejected_parquet = DATA / 'hdma_rejected_raw.parquet.gzip'


accepted_recovered = pd.read_parquet(hdma_accepted_parquet, columns=temp_num_cols + temp_text_cols)
rejected_recovered = pd.read_parquet(hdma_rejected_parquet, columns=temp_num_cols + temp_text_cols)

In [3]:
accepted_recovered.head(10)

,activity_year,action_taken,preapproval,loan_purpose,loan_amount,loan_to_value_ratio,loan_term,income,debt_to_income_ratio,derived_loan_product_type,applicant_credit_score_type,co-applicant_credit_score_type,denial_reason-1
0,2023.0,1.0,2.0,1.0,75000.0,80.00000,360,180.0,44,Conventional:First Lien,3.0,10.0,10.0
1,2023.0,1.0,2.0,1.0,445000.0,84.00000,360,153.0,37,Conventional:First Lien,1.0,9.0,10.0
2,2023.0,1.0,2.0,1.0,145000.0,96.50000,360,73.0,42,FHA:First Lien,2.0,10.0,10.0
3,2023.0,1.0,2.0,1.0,425000.0,75.00000,360,NaN,46,Conventional:First Lien,2.0,9.0,10.0
4,2023.0,1.0,2.0,1.0,425000.0,100.88000,360,84.0,49,Conventional:First Lien,2.0,9.0,10.0
5,2023.0,1.0,2.0,1.0,325000.0,80.00000,360,189.0,30%-<36%,Conventional:First Lien,3.0,10.0,10.0
6,2023.0,1.0,2.0,1.0,305000.0,95.00000,360,65.0,46,FHA:First Lien,3.0,10.0,10.0
7,2023.0,1.0,1.0,1.0,175000.0,100.00000,360,61.0,44,FHA:First Lien,1.0,10.0,10.0
8,2023.0,1.0,2.0,1.0,125000.0,50.00000,360,132.0,20%-<30%,Conventional:First Lien,3.0,9.0,10.0
9,2023.0,1.0,2.0,1.0,275000.0,87.86900,360,100.0,39,FHA:First Lien,3.0,9.0,10.0


In [4]:
rejected_recovered.head(10)

,activity_year,action_taken,preapproval,loan_purpose,loan_amount,loan_to_value_ratio,loan_term,income,debt_to_income_ratio,derived_loan_product_type,applicant_credit_score_type,co-applicant_credit_score_type,denial_reason-1
0,2023.0,3.0,2.0,1.0,545000.0,80.00000,360,110.0,50%-60%,Conventional:First Lien,2.0,9.0,1.0
1,2023.0,3.0,2.0,1.0,315000.0,98.97400,360,65.0,50%-60%,FHA:First Lien,3.0,9.0,4.0
2,2023.0,3.0,2.0,1.0,255000.0,96.50000,360,39.0,>60%,FHA:First Lien,2.0,10.0,1.0
3,2023.0,3.0,2.0,1.0,425000.0,90.00000,360,85.0,>60%,Conventional:First Lien,2.0,9.0,1.0
4,2023.0,3.0,2.0,32.0,425000.0,78.87900,360,158.0,30%-<36%,FHA:First Lien,1.0,9.0,1.0
5,2023.0,3.0,2.0,32.0,215000.0,84.70600,360,92.0,>60%,VA:First Lien,3.0,10.0,1.0
6,2023.0,3.0,2.0,1.0,245000.0,96.50000,360,48.0,50%-60%,FHA:First Lien,1.0,10.0,1.0
7,2023.0,3.0,2.0,32.0,565000.0,76.08700,360,134.0,50%-60%,FHA:First Lien,1.0,9.0,3.0
8,2023.0,3.0,2.0,1.0,415000.0,75.00000,360,177.0,50%-60%,Conventional:First Lien,3.0,9.0,1.0
9,2023.0,3.0,2.0,32.0,365000.0,40.78200,360,47.0,>60%,Conventional:First Lien,2.0,10.0,1.0


**Fix the HDMA accepted tables**
- Instead of replacing data with mode, find the median according to certain ranges
    - If a range is provided, find the mean of all values that fall within that range in that column and replace ranges/NAs with that, etc

In [65]:
accepted_copy = accepted_recovered.copy()

#Fix the income to be in thousands value
accepted_copy['income'] = accepted_copy['income'] * 1000

#Fix any columns that have values in the form 'Exempt'
exempt_cols = ['debt_to_income_ratio', 'income', 'loan_term', 'loan_to_value_ratio']
for columns in exempt_cols:
    accepted_copy[columns] = accepted_copy[columns].replace({'Exempt': None})

accepted_copy["denial_reason-1"] = accepted_copy["denial_reason-1"].replace(1111, 1)

In [66]:
#Replace any entries that are ranges in the column with whole numbers
accepted_copy['debt_to_income_ratio'] = accepted_copy['debt_to_income_ratio'].replace(">60%", 62)
accepted_copy['debt_to_income_ratio'] = accepted_copy['debt_to_income_ratio'].replace("<20%", 19)
accepted_copy['debt_to_income_ratio'] = accepted_copy['debt_to_income_ratio'].replace("50%-60%", 55)
accepted_copy['debt_to_income_ratio'] = accepted_copy['debt_to_income_ratio'].replace("20%-<30%", (29+20)//2)
accepted_copy['debt_to_income_ratio'] = accepted_copy['debt_to_income_ratio'].replace("30%-<36%", (35+30)//2)

In [ ]:
#convert all appropriate columns to int
int_values = [
        'activity_year', 'action_taken', 'preapproval', 'loan_purpose', 'loan_amount', 'loan_term', 'applicant_credit_score_type',
         'co-applicant_credit_score_type', 'debt_to_income_ratio', 'denial_reason_1'
    ]

#replace int values with mode since the values are all whole numbers
for column in int_values:
    #print(f"fixing {accepted_copy[column].isnull().sum()} nulls to {accepted_copy[column].value_counts().reset_index().iat[0, 0]}")
    accepted_copy[column] = accepted_copy[column].fillna(accepted_copy[column].value_counts().reset_index().iat[0, 0])
    try:
        accepted_copy[column] = accepted_copy[column].astype('int64')

    except Exception as e:
        print(f"skipping int conversion: {column} due to error: {e}") 
        continue

accepted_copy.head(10)

,activity_year,action_taken,preapproval,loan_purpose,loan_amount,loan_to_value_ratio,loan_term,income,debt_to_income_ratio,derived_loan_product_type,applicant_credit_score_type,co-applicant_credit_score_type,denial_reason-1
0,2023,1,2,1,75000,80.00000,360,180000.0,44,Conventional:First Lien,3,10,10
1,2023,1,2,1,445000,84.00000,360,153000.0,37,Conventional:First Lien,1,9,10
2,2023,1,2,1,145000,96.50000,360,73000.0,42,FHA:First Lien,2,10,10
3,2023,1,2,1,425000,75.00000,360,NaN,46,Conventional:First Lien,2,9,10
4,2023,1,2,1,425000,100.88000,360,84000.0,49,Conventional:First Lien,2,9,10
5,2023,1,2,1,325000,80.00000,360,189000.0,32,Conventional:First Lien,3,10,10
6,2023,1,2,1,305000,95.00000,360,65000.0,46,FHA:First Lien,3,10,10
7,2023,1,1,1,175000,100.00000,360,61000.0,44,FHA:First Lien,1,10,10
8,2023,1,2,1,125000,50.00000,360,132000.0,24,Conventional:First Lien,3,9,10
9,2023,1,2,1,275000,87.86900,360,100000.0,39,FHA:First Lien,3,9,10


In [57]:
accepted_copy[int_values].isna().sum()

activity_year                     0
action_taken                      0
preapproval                       0
loan_purpose                      0
loan_amount                       0
loan_term                         0
applicant_credit_score_type       0
debt_to_income_ratio              0
co-applicant_credit_score_type    0
denial_reason-1                   0
dtype: int64

In [58]:
#convert all columns to floats
accepted_float = accepted_copy.copy()
float_values = [
        'loan_to_value_ratio', 'income'
    ]

#Fix any float values in df
for column in float_values:
    try:
        accepted_float[column] = pd.to_numeric(accepted_float[column], errors='ignore')
    except Exception as e:
        print(f"error when trying to convert {column} to int: {e}")
        continue
    
    try:
        mean = accepted_float[column].mean()
        print(f"mean of {column} is {mean}")
        accepted_float[column] = accepted_float[column].fillna(mean)
    except Exception as e:
        print(f"skipping float mean: {column} due to error: {e}") 
        continue

string_values = [
    'derived_loan_product_type'
]

for column in string_values:
        accepted_float[column] = accepted_float[column].fillna(accepted_float[column].value_counts().reset_index().iat[0, 0])

'''
#Convert any ranges in the debt_income cat into the middle of that range, or leave it at that range if its too broad
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace(">60%", 60)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("50%-60%", 55)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("20%-<30%", (29+20)/2)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("30%-<36%", (35+30)/2)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("<20%", 20)
'''

#accepted_float[float_values].dtypes
accepted_float.isna().sum()

C:\Users\Saul\AppData\Local\Temp\ipykernel_34628\1630531970.py:10: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  accepted_float[column] = pd.to_numeric(accepted_float[column], errors='ignore')


mean of loan_to_value_ratio is 83.44945297968425
mean of income is 184638.19623032882


activity_year                     0
action_taken                      0
preapproval                       0
loan_purpose                      0
loan_amount                       0
loan_to_value_ratio               0
loan_term                         0
income                            0
debt_to_income_ratio              0
derived_loan_product_type         0
applicant_credit_score_type       0
co-applicant_credit_score_type    0
denial_reason-1                   0
dtype: int64

In [ ]:
len(accepted_float)

3452631

In [6]:
import random

def remove_extremes(df = pd.DataFrame()):
    """
    Function that removes nonsense values (negative income or loan amount, etc)
    """

    values = [
        'loan_amount', 'loan_term', 'debt_to_income_ratio', 'loan_to_value_ratio', 'income'
    ]

    filtered = df.copy()

    for col in values:
        filtered = filtered[filtered[col] >= 0]
        
    return filtered

def clean_data(df = pd.DataFrame(), sample = bool, seed = int): 
    """
    Helper function made to clean the HDMA dataframes
    """
    int_values = [
        'activity_year', 'action_taken', 'preapproval', 'loan_purpose', 'loan_amount', 'loan_term', 'applicant_credit_score_type',
        'co-applicant_credit_score_type', 'debt_to_income_ratio'
    ]

    float_values = [
        'loan_to_value_ratio', 'income'
    ]

    string_values = [
        'derived_loan_product_type'
    ]

    #Fix the income to be in thousands value
    df['income'] = df['income'] * 1000

    #Fix any columns that have values in the form 'Exempt'
    exempt_cols = ['debt_to_income_ratio', 'income', 'loan_term', 'loan_to_value_ratio']
    for columns in exempt_cols:
        df[columns] = df[columns].replace({'Exempt': None})
    
    #Fix the 1111's in denial reason
    if "denial_reason_1" in df.columns:
        df["denial_reason_1"] = df["denial_reason_1"].replace(1111, 1)
        df["denial_reason_1"] = df["denial_reason_1"].fillna(df["denial_reason_1"].value_counts().reset_index().iat[0, 0])
        try:
            df["denial_reason_1"] = df["denial_reason_1"].astype('int64')

        except Exception as e:
            print(f"skipping int conversion: denial_reason_1 due to error: {e}") 
    
    #Convert any ranges in the debt_income cat into the middle of that range, or leave it at that range if its too broad
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace(">60%", 62)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("<20%", 19)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("50%-60%", 55)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("20%-<30%", (29+20)//2)
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'].replace("30%-<36%", (35+30)//2)
    
    #replace int values with mode since the values are all whole numbers
    for column in int_values:
        df[column] = df[column].fillna(df[column].value_counts().reset_index().iat[0, 0])
        try:
            df[column] = df[column].astype('int64')

        except Exception as e:
            print(f"skipping int conversion: {column} due to error: {e}") 
            continue
        
    #Fix float columns by replacing NA's with mean 
    for column in float_values:
        try:
            df[column] = pd.to_numeric(df[column], errors='ignore')
        except Exception as e:
            print(f"error when trying to convert {column} to int: {e}")
            continue
        
        try:
            mean = df[column].mean()
            print(f"mean of {column} is {mean}")
            df[column] = df[column].fillna(mean)
        except Exception as e:
            print(f"skipping float mean: {column} due to error: {e}") 
            continue

    #Fix string columns by replacing with mode
    for column in string_values:
        df[column] = df[column].fillna(df[column].value_counts().reset_index().iat[0, 0])

    cleaned = remove_extremes(df.copy())

    try:
        if sample is True and seed != 0:
                #Create a random seed for sampling the large dataset if no seed is provided
                return cleaned.sample(200000, random_state = seed)
        elif sample is True:
                seed = random.randint(0,999)
                print(f"Random seed to replicate accepted loans df: {seed}")
                return cleaned.sample(200000, random_state = seed)
        else:
            print(f"Sample input was set --> {sample}, thus the entire dataset will be returned")
            return cleaned
    except Exception as e:
        print(f"Error when attempting to get the dataframe cleaned up : {e}")
        return

In [7]:
rejected_recovered = rejected_recovered.rename(columns={"denial_reason-1":"denial_reason_1"})

cleaned_rejected = clean_data(rejected_recovered.copy(), sample=False)

cleaned_accepted = clean_data(accepted_recovered.copy(), sample=False)

C:\Users\Saul\AppData\Local\Temp\ipykernel_26004\2098182582.py:74: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[column] = pd.to_numeric(df[column], errors='ignore')


mean of loan_to_value_ratio is 5712.198193689505
mean of income is 206675.92802995408
Sample input was set --> False, thus the entire dataset will be returned


KeyboardInterrupt: 

In [8]:
test_rej = remove_extremes(cleaned_rejected.copy())
test_acc = remove_extremes(cleaned_accepted.copy())

In [9]:
print(f"len rej raw:{len(rejected_recovered)} ---clean--> {len(cleaned_rejected)}")
print(f"len acc clean:{len(accepted_recovered)} ---clean--> {len(cleaned_accepted)}")

print(f"len rej raw:{len(cleaned_rejected)} ---clean--> {len(test_rej)}")
print(f"len acc raw:{len(cleaned_accepted)} ---clean--> {len(test_acc)}")

len rej raw:2037095 ---clean--> 2031108
len acc clean:3452631 ---clean--> 3452490
len rej raw:2031108 ---clean--> 2031108
len acc raw:3452490 ---clean--> 3452490


In [8]:
from ETL.ingestion import data_ingestion_hdma
cc = data_ingestion_hdma.clean_hdma_accepted(sample = False)
cr = data_ingestion_hdma.clean_hdma_rejected(sample = False)

Sample input was set --> False, thus the entire dataset will be returned
Sample input was set --> False, thus the entire dataset will be returned


In [11]:
#confirm types of accepted
cc.dtypes

activity_year                              int64
action_taken                               int64
preapproval                                int64
loan_purpose                               int64
loan_amount                                int64
loan_term                                  int64
applicant_credit_score_type                int64
co_applicant_credit_score_type             int64
loan_to_value_ratio                      float64
income                                   float64
debt_to_income_ratio                       int64
derived_loan_product_type         string[python]
dtype: object

In [9]:
cc.max()

activity_year                                    2023
action_taken                                        1
preapproval                                         2
loan_purpose                                        1
loan_amount                                 383475000
loan_term                                         506
applicant_credit_score_type                      1111
co_applicant_credit_score_type                   1111
loan_to_value_ratio                           2904.47
income                                  73490000000.0
debt_to_income_ratio                               62
derived_loan_product_type         VA:Subordinate Lien
dtype: object

In [12]:
#confirm types of rejected
cr.dtypes

activity_year                              int64
action_taken                               int64
preapproval                                int64
loan_purpose                               int64
loan_amount                                int64
loan_term                                  int64
applicant_credit_score_type                int64
co_applicant_credit_score_type             int64
denial_reason_1                            int64
loan_to_value_ratio                      float64
income                                   float64
debt_to_income_ratio                       int64
derived_loan_product_type         string[python]
dtype: object

In [10]:
cr.max()

activity_year                                    2023
action_taken                                        3
preapproval                                         2
loan_purpose                                       32
loan_amount                              232323235000
loan_term                                         685
applicant_credit_score_type                      1111
co_applicant_credit_score_type                   1111
denial_reason_1                                     9
loan_to_value_ratio                      4025000000.0
income                                 132000000000.0
debt_to_income_ratio                               62
derived_loan_product_type         VA:Subordinate Lien
dtype: object

In [13]:
cc.head(10)

,activity_year,action_taken,preapproval,loan_purpose,loan_amount,loan_term,applicant_credit_score_type,co_applicant_credit_score_type,loan_to_value_ratio,income,debt_to_income_ratio,derived_loan_product_type
0,2023,1,2,1,75000,360,3,10,80.000,180000.00000,44,Conventional:First Lien
1,2023,1,2,1,445000,360,1,9,84.000,153000.00000,37,Conventional:First Lien
2,2023,1,2,1,145000,360,2,10,96.500,73000.00000,42,FHA:First Lien
3,2023,1,2,1,425000,360,2,9,75.000,184638.19623,46,Conventional:First Lien
4,2023,1,2,1,425000,360,2,9,100.880,84000.00000,49,Conventional:First Lien
5,2023,1,2,1,325000,360,3,10,80.000,189000.00000,32,Conventional:First Lien
6,2023,1,2,1,305000,360,3,10,95.000,65000.00000,46,FHA:First Lien
7,2023,1,1,1,175000,360,1,10,100.000,61000.00000,44,FHA:First Lien
8,2023,1,2,1,125000,360,3,9,50.000,132000.00000,24,Conventional:First Lien
9,2023,1,2,1,275000,360,3,9,87.869,100000.00000,39,FHA:First Lien


In [14]:
cr.head(10)


,activity_year,action_taken,preapproval,loan_purpose,loan_amount,loan_term,applicant_credit_score_type,co_applicant_credit_score_type,denial_reason_1,loan_to_value_ratio,income,debt_to_income_ratio,derived_loan_product_type
0,2023,3,2,1,545000,360,2,9,1,80.000,110000.0,55,Conventional:First Lien
1,2023,3,2,1,315000,360,3,9,4,98.974,65000.0,55,FHA:First Lien
2,2023,3,2,1,255000,360,2,10,1,96.500,39000.0,62,FHA:First Lien
3,2023,3,2,1,425000,360,2,9,1,90.000,85000.0,62,Conventional:First Lien
4,2023,3,2,32,425000,360,1,9,1,78.879,158000.0,32,FHA:First Lien
5,2023,3,2,32,215000,360,3,10,1,84.706,92000.0,62,VA:First Lien
6,2023,3,2,1,245000,360,1,10,1,96.500,48000.0,55,FHA:First Lien
7,2023,3,2,32,565000,360,1,9,3,76.087,134000.0,55,FHA:First Lien
8,2023,3,2,1,415000,360,3,9,1,75.000,177000.0,55,Conventional:First Lien
9,2023,3,2,32,365000,360,2,10,1,40.782,47000.0,62,Conventional:First Lien


In [ ]:
cc_col = list(cc.columns)
cr_col = list(cr.columns)

print(cc_col)
print(cr_col)

['activity_year', 'action_taken', 'preapproval', 'loan_purpose', 'loan_amount', 'loan_term', 'applicant_credit_score_type', 'co_applicant_credit_score_type', 'loan_to_value_ratio', 'income', 'debt_to_income_ratio', 'derived_loan_product_type']
['activity_year', 'action_taken', 'preapproval', 'loan_purpose', 'loan_amount', 'loan_term', 'applicant_credit_score_type', 'co_applicant_credit_score_type', 'denial_reason_1', 'loan_to_value_ratio', 'income', 'debt_to_income_ratio', 'derived_loan_product_type']
